# ADY201m - AI, DATA SCIENCE WITH PYTHON & SQL
### FINAL REPORT

---

### THÔNG TIN DỰ ÁN
- **Tên dự án:** Dự đoán học lực & Cảnh báo sớm sinh viên có nguy cơ học lại (Student Performance Analytics)
- **Mã lớp:** AI2102
- **Tên nhóm:** Nhóm 6
- **Thành viên nhóm:**
  1. Hoàng Văn Đức - HE210214 - Thành Viên
  2. Phạm Huy Vũ - HE210285 - Thành viên
  3. Nguyễn Tuấn Anh - HE210230 - Thành Viên
  4. Nguyễn Đình An - HE210153 - Thành Viên
- **Nguồn Dataset:** Open University Learning Analytics Dataset (OULAD) – UCI Machine Learning Repository, https://archive.ics.uci.edu/dataset/349 (DOI: 10.24432/C5KK69), giấy phép CC-BY 4.0

### 1.1: Bối cảnh bài toán và lý do thực hiện
- **Bối cảnh bài toán:** Ở các trường đại học, việc sinh viên trượt môn hoặc bỏ dở học phần gây thiệt hại cho cả hai phía. Sinh viên mất thêm học phí và thời gian, có nguy cơ chậm tốt nghiệp. Nhà trường thì giảm tỷ lệ tốt nghiệp đúng hạn và tốn thêm nguồn lực mở lớp học lại. Hiện nay việc phát hiện sinh viên gặp khó khăn chủ yếu dựa vào điểm giữa kỳ hoặc cuối kỳ, tức là khi đã quá muộn để can thiệp. Trong khi đó, hệ thống học trực tuyến (LMS/VLE) ghi lại hàng ngày dấu vết hành vi học tập: đăng nhập, truy cập tài liệu, nộp bài. Đây là nguồn dữ liệu có sẵn nhưng chưa được khai thác.
- **Vấn đề chính cần giải quyết:** Xác định những sinh viên có nguy cơ cao sẽ trượt (Fail) hoặc rút khỏi học phần (Withdrawn), tức là có nguy cơ phải học lại **ngay từ khi có được kết quả của các bài kiểm tra chiếm 40-50% trọng số điểm môn học** . Việc dự đoán chỉ dựa trên thông tin nền tảng, lịch sử học lại, hành vi trên VLE và kết quả các bài kiểm tra đầu tiên.
- **Giá trị/ Ý nghĩa đem lại:** Danh sách cảnh báo sớm giúp cố vấn học tập và giảng viên ưu tiên nguồn lực hỗ trợ có hạn (gọi điện, phụ đạo, tư vấn giảm tải) cho đúng người, vào lúc còn đủ thời gian để thay đổi kết quả. Các hệ thống tương tự như Course Signals tại Đại học Purdue (Arnold & Pistilli, 2012) và OU Analyse tại The Open University (Hlosta et al., 2017) đã chứng minh tính khả thi. Dự án dùng bộ dữ liệu OULAD của The Open University (Anh), gồm 32.593 lượt đăng ký học phần của sinh viên đại học.

### 1.2: Các câu hỏi cốt lõi
- **Câu hỏi 1:** Những yếu tố nào quan sát được trước mốc dự đoán dự báo mạnh nhất việc sinh viên trượt hoặc bỏ học phần? Các yếu tố được xem xét gồm mức độ tương tác VLE, kết quả các bài kiểm tra đầu tiên và số lần đã học lại.
- **Câu hỏi 2:** Với nguồn lực tư vấn có hạn, mô hình cần đạt Recall bao nhiêu để phát hiện phần lớn sinh viên có nguy cơ?

### 1.3: Chi tiết bài toán
- **Problem Type:** Học có giám sát (Supervised Learning), phân loại 2 nhóm (Binary Classification).
- **Đơn vị phân tích:** 1 dòng = 1 sinh viên × 1 lớp học phần (`code_module` + `code_presentation`).
- **Thời điểm dự đoán (cut-off):** Sau khi có đủ 40-50% trọng số điểm môn học kể từ khi học phần bắt đầu. Có hai lý do chọn mốc này:
  - Khi đạt được tầm 40-50% trọng số điểm của môn học, đã có ít nhất 2 bài kiểm tra được hoàn thành và khoảng 60 đến 120 ngày học để đánh giá dựa trên điểm số và lượng truy cập vào tài liệu học tập.
  - Khi đó vẫn còn khoảng 45% thời lượng học phần để can thiệp.
- **Target Variable (y):** `at_risk` có hai giá trị:
  - `1` (có nguy cơ học lại): `final_result` là `Fail` hoặc `Withdrawn`.
  - `0` (an toàn): `final_result` là `Pass` hoặc `Distinction`.
- **Expected Feature Variables (X):**
  - Lịch sử học tập: số lần đã học học phần này (`num_of_prev_attempts`).
  - Hành vi VLE đến thời điểm dự đoán: tổng click, số ngày hoạt động, số tài nguyên truy cập, click trước khai giảng.
  - Đánh giá đến thời điểm dự đoán: số bài đến hạn, số bài đã nộp, điểm trung bình, số bài nộp trễ.
  - **Không dùng** các thông tin chỉ có sau mốc cut-off, như ngày rút học phần và điểm các bài sau thời điểm dự đoán, để tránh rò rỉ thông tin tương lai.
- **Evaluation Metric:**
  - Chỉ số chính: **Recall** của lớp `at_risk = 1` và **F1-Score**. Lý do là bỏ sót một sinh viên cần hỗ trợ (False Negative) tốn kém hơn nhiều so với một cuộc gọi tư vấn thừa (False Positive).
  - Chỉ số phụ: **PR-AUC** và **ROC-AUC** để so sánh mô hình độc lập với ngưỡng phân loại.

# PART 2: THU THẬP DỮ LIỆU

### 2.1 Nguồn dữ liệu

| Thuộc tính | Mô tả |
|:---|:---|
| **Tên dataset** | Open University Learning Analytics Dataset (OULAD) |
| **Công bố gốc** | Kuzilek, Hlosta, Zdrahal (2017). *Scientific Data* 4:170171. DOI 10.1038/sdata.2017.171 |
| **Repository** | UCI ML Repository, id 349 (DOI 10.24432/C5KK69) |
| **Giấy phép** | CC-BY 4.0 |
| **Thời gian** | 4 kỳ: 2013B, 2013J, 2014B, 2014J (B=Feb, J=Oct) |
| **Phạm vi** | 32.593 lượt ghi danh × 7 bảng quan hệ |

### 2.2 Crawl dữ liệu

Tải file zip OULAD từ UCI và giải nén ra **7 bảng CSV** vào `data/raw/`. Nếu đã có đủ file thì bỏ qua bước tải.

In [ ]:
import zipfile, urllib.request
from pathlib import Path

URL = 'https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip'
RAW = Path('data/raw')
TABLES = ['courses', 'assessments', 'vle', 'studentInfo',
          'studentRegistration', 'studentAssessment', 'studentVle']

if not all((RAW / f'{t}.csv').exists() for t in TABLES):
    RAW.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(URL, RAW / 'oulad.zip')
    zipfile.ZipFile(RAW / 'oulad.zip').extractall(RAW)

for t in TABLES:
    print(f'{t + ".csv":26s} {(RAW / f"{t}.csv").stat().st_size / 1e6:8.1f} MB')

### 2.3 Tích hợp dữ liệu bằng SQL (magic `%%sql`)

Dùng **jupysql** + **DuckDB**: DuckDB đọc thẳng file CSV nên không cần bước nạp dữ liệu vào database. Pipeline gồm 5 bước, mọi đặc trưng chỉ dùng dữ liệu **trước `cutoff_day`** để không nhìn tương lai.

In [ ]:
import duckdb
conn = duckdb.connect()
%load_ext sql
%sql conn
%config SqlMagic.displaylimit = 5

**Bước 0.** Khai báo 5 bảng CSV cần dùng dưới dạng view (`?` trong dữ liệu gốc là NULL). `courses` và `vle` không cần cho bài toán.

In [ ]:
%%sql
CREATE OR REPLACE VIEW assessments         AS SELECT * FROM read_csv('data/raw/assessments.csv',         nullstr = '?');
CREATE OR REPLACE VIEW studentInfo         AS SELECT * FROM read_csv('data/raw/studentInfo.csv',         nullstr = '?');
CREATE OR REPLACE VIEW studentRegistration AS SELECT * FROM read_csv('data/raw/studentRegistration.csv', nullstr = '?');
CREATE OR REPLACE VIEW studentAssessment   AS SELECT * FROM read_csv('data/raw/studentAssessment.csv',   nullstr = '?');
CREATE OR REPLACE VIEW studentVle          AS SELECT * FROM read_csv('data/raw/studentVle.csv',          nullstr = '?');

**Bước 1 — `module_cutoff`.** Sắp các bài kiểm tra theo ngày đến hạn, cộng dồn trọng số. `cutoff_day` là ngày đầu tiên tổng trọng số đạt ≥ 40%. Cutoff tính riêng cho từng lớp vì mỗi môn có lịch kiểm tra khác nhau.

In [ ]:
%%sql
CREATE OR REPLACE TABLE module_cutoff AS
SELECT code_module, code_presentation, MIN(date) AS cutoff_day
FROM (
    SELECT *, SUM(weight) OVER (PARTITION BY code_module, code_presentation
                                ORDER BY date ROWS UNBOUNDED PRECEDING) AS cum_weight
    FROM assessments
    WHERE date IS NOT NULL AND weight > 0
)
WHERE cum_weight >= 40
GROUP BY code_module, code_presentation;

SELECT code_module, MIN(cutoff_day) AS min_cutoff, MAX(cutoff_day) AS max_cutoff
FROM module_cutoff GROUP BY code_module ORDER BY code_module;

**Bước 2 — `assess_due`.** Các bài kiểm tra (trọng số > 0) đến hạn trước cutoff.

In [ ]:
%%sql
CREATE OR REPLACE TABLE assess_due AS
SELECT a.*
FROM assessments a
JOIN module_cutoff c USING (code_module, code_presentation)
WHERE a.date < c.cutoff_day AND a.weight > 0;

**Bước 3 — `vle_agg`.** Tổng hợp click VLE của mỗi sinh viên trước cutoff. Click có `date < 0` là click trước ngày khai giảng.

In [ ]:
%%sql
CREATE OR REPLACE TABLE vle_agg AS
SELECT v.code_module, v.code_presentation, v.id_student,
       SUM(v.sum_click)                                      AS total_clicks,
       COUNT(DISTINCT v.date)                                AS active_days,
       COUNT(DISTINCT v.id_site)                             AS n_resources,
       SUM(CASE WHEN v.date < 0 THEN v.sum_click ELSE 0 END) AS clicks_before_start
FROM studentVle v
JOIN module_cutoff c USING (code_module, code_presentation)
WHERE v.date < c.cutoff_day
GROUP BY ALL;

**Bước 4 — `assess_agg`.** Số bài đã nộp, điểm trung bình và số bài nộp trễ của mỗi sinh viên trước cutoff.

In [ ]:
%%sql
CREATE OR REPLACE TABLE assess_agg AS
SELECT d.code_module, d.code_presentation, s.id_student,
       COUNT(*)                                                   AS n_submitted,
       AVG(s.score)                                               AS avg_score,
       SUM(CASE WHEN s.date_submitted > d.date THEN 1 ELSE 0 END) AS n_late
FROM studentAssessment s
JOIN assess_due d    USING (id_assessment)
JOIN module_cutoff c USING (code_module, code_presentation)
WHERE s.date_submitted < c.cutoff_day
GROUP BY ALL;

**Bước 5 — `df_raw`.** Ghép tất cả theo `(code_module, code_presentation, id_student)`:
- `LEFT JOIN` để giữ sinh viên không click / không nộp bài nào: đây là nhóm nguy cơ cao nhất, `INNER JOIN` sẽ xoá họ.
- Bỏ biến nhân khẩu học (lý do công bằng) và `date_unregistration` (thông tin tương lai).
- Loại sinh viên đã rút trước cutoff (không còn can thiệp được) và sinh viên đăng ký sau cutoff (chưa có dữ liệu).

In [ ]:
%%sql
CREATE OR REPLACE TABLE df_raw AS
SELECT si.code_module, si.code_presentation, si.id_student,
       si.num_of_prev_attempts, si.studied_credits, sr.date_registration, c.cutoff_day,
       COALESCE(v.total_clicks, 0)        AS total_clicks,
       COALESCE(v.active_days, 0)         AS active_days,
       COALESCE(v.n_resources, 0)         AS n_resources,
       COALESCE(v.clicks_before_start, 0) AS clicks_before_start,
       (SELECT COUNT(*) FROM assess_due d
         WHERE d.code_module = si.code_module
           AND d.code_presentation = si.code_presentation) AS n_assess_due,
       COALESCE(a.n_submitted, 0)         AS n_submitted,
       a.avg_score,
       COALESCE(a.n_late, 0)              AS n_late,
       si.final_result,
       CASE WHEN si.final_result IN ('Fail', 'Withdrawn') THEN 1 ELSE 0 END AS at_risk
FROM studentInfo si
JOIN studentRegistration sr USING (code_module, code_presentation, id_student)
JOIN module_cutoff c        USING (code_module, code_presentation)
LEFT JOIN vle_agg v         USING (code_module, code_presentation, id_student)
LEFT JOIN assess_agg a      USING (code_module, code_presentation, id_student)
WHERE (sr.date_unregistration IS NULL OR sr.date_unregistration > c.cutoff_day)
  AND (sr.date_registration   IS NULL OR sr.date_registration  <= c.cutoff_day)
ORDER BY si.code_module, si.code_presentation, si.id_student;

SELECT COUNT(*) AS n_rows, ROUND(AVG(at_risk), 3) AS at_risk_rate FROM df_raw;

Xuất bảng tích hợp ra CSV để kiểm chứng và dùng cho các phần sau.

In [ ]:
%%sql
COPY df_raw TO 'data/integrated/Intergrated_data.csv' (HEADER);

**Nhận xét.** `cutoff_day` khác nhau rõ giữa các môn (EEE khoảng 68 ngày, GGG khoảng 222 ngày) nên không dùng được một ngày cố định. Sau khi lọc còn **25.219 dòng × 17 cột**, at_risk khoảng 39%.

# PART 3: LÀM SẠCH & TIỀN XỬ LÝ

### 3.1 Từ điển dữ liệu (Data Dictionary)

**7 bảng gốc (phần 2.2):**

| Bảng | Số dòng | Khoá | Nội dung chính |
|:---|---:|:---|:---|
| `courses` | 22 | `(module, presentation)` | Danh sách lớp + độ dài kỳ (ngày) |
| `assessments` | 206 | `id_assessment` | Loại bài (TMA/CMA/Exam), ngày đến hạn, trọng số % |
| `vle` | 6.364 | `id_site` | Tài nguyên học + activity_type |
| `studentInfo` | 32.593 | `(module, pres, id_student)` | **Thông tin SV + `final_result`** |
| `studentRegistration` | 32.593 | như trên | `date_registration`, `date_unregistration` |
| `studentAssessment` | 173.912 | `(id_assessment, id_student)` | `score` + `date_submitted` |
| `studentVle` | **10.655.280** | long format | Log click hằng ngày: `(id_site, date, sum_click)` |

Cột `date` trong mọi bảng là **số ngày tính từ ngày khai giảng** (date < 0 = trước khai giảng).

**Bảng tích hợp (phần 2.3):** sinh tự động từ dữ liệu và xuất ra `data/integrated/data_dictionary.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
RNG = 42

df = pd.read_csv('data/integrated/Intergrated_data.csv')

meaning = {
    'code_module':          'Mã môn học',
    'code_presentation':    'Kỳ học (năm + B=tháng 2 / J=tháng 10)',
    'id_student':           'Mã sinh viên',
    'num_of_prev_attempts': 'Số lần đã học môn này trước đó',
    'studied_credits':      'Tổng số tín chỉ đang học',
    'date_registration':    'Ngày đăng ký so với ngày khai giảng',
    'cutoff_day':           'Ngày cảnh báo (tích luỹ >= 40% trọng số)',
    'total_clicks':         'Tổng click VLE trước cutoff',
    'active_days':          'Số ngày có hoạt động VLE trước cutoff',
    'n_resources':          'Số tài nguyên VLE khác nhau đã truy cập',
    'clicks_before_start':  'Click VLE trước ngày khai giảng',
    'n_assess_due':         'Số bài kiểm tra đến hạn trước cutoff',
    'n_submitted':          'Số bài đã nộp trước cutoff',
    'avg_score':            'Điểm trung bình các bài đã nộp',
    'n_late':               'Số bài nộp trễ',
    'final_result':         'Kết quả cuối kỳ (chỉ dùng để tạo target)',
    'at_risk':              'Target: 1 = Fail/Withdrawn, 0 = Pass/Distinction',
}
data_dict = pd.DataFrame({
    'dtype':     df.dtypes.astype(str),
    'n_missing': df.isna().sum(),
    'n_unique':  df.nunique(),
    'example':   df.iloc[0],
    'meaning':   pd.Series(meaning),
})
data_dict.to_csv('data/integrated/data_dictionary.csv', index_label='column')
print(df.shape)
data_dict

### 3.2 Xử lý giá trị thiếu (Missing Values)

Chỉ 2 cột có missing, mỗi cột có nguyên nhân riêng nên xử lý riêng:

| Cột | Nguyên nhân | Xử lý |
|---|---|---|
| `date_registration` (7 dòng) | Lỗi nhập liệu hiếm | Điền median theo lớp `(code_module, code_presentation)` vì mỗi lớp có cửa sổ đăng ký riêng. |
| `avg_score` (3.730 dòng) | Sinh viên chưa nộp bài nào (`n_submitted = 0`) | Điền **0** và tạo 2 biến cờ để mô hình phân biệt hai tình huống (bên dưới). |
| `avg_score` (8 dòng) | Đã nộp nhưng chưa được chấm | Điền median theo lớp. |

Hai biến cờ (flag, 0/1):
- `dropout_no_activity`: không nộp bài **và** không click VLE, tức đã bỏ học hẳn.
- `active_but_no_submission`: không nộp bài **nhưng** vẫn vào VLE, tức đang trì hoãn.

In [ ]:
df.isna().sum()[lambda s: s > 0]

In [ ]:
cls = ['code_module', 'code_presentation']

df['dropout_no_activity']      = ((df['n_submitted'] == 0) & (df['total_clicks'] == 0)).astype(int)
df['active_but_no_submission'] = ((df['n_submitted'] == 0) & (df['total_clicks'] > 0)).astype(int)

df.loc[df['n_submitted'] == 0, 'avg_score'] = 0
for c in ['date_registration', 'avg_score']:
    df[c] = df[c].fillna(df.groupby(cls)[c].transform('median'))

print('Còn missing:', df.isna().sum().sum())
df.groupby('dropout_no_activity')['at_risk'].agg(['count', 'mean']), \
df.groupby('active_but_no_submission')['at_risk'].agg(['count', 'mean'])

**Nhận xét.** Nhóm `dropout_no_activity = 1` (411 SV) có tỷ lệ nguy cơ khoảng **99%**, nhóm `active_but_no_submission = 1` (3.311 SV) khoảng **54%**, đều cao hơn hẳn mức trung bình 39%. Hai nhóm có mức rủi ro khác nhau nên tách thành 2 cờ là hợp lý.

### 3.3 Xử lý dữ liệu trùng lặp (Duplicates)

Kiểm tra trùng toàn bộ dòng và trùng theo đơn vị phân tích `(code_module, code_presentation, id_student)`. Nếu có trùng theo đơn vị phân tích thì phép JOIN trong SQL bị sai.

In [ ]:
key = ['code_module', 'code_presentation', 'id_student']
print('Trùng toàn bộ dòng       :', df.duplicated().sum())
print('Trùng theo đơn vị phân tích:', df.duplicated(subset=key).sum())
df = df.drop_duplicates(subset=key)
print('Shape:', df.shape)

**Nhận xét.** Không có dòng trùng, phép JOIN trong SQL đúng: mỗi sinh viên chỉ xuất hiện một lần trong mỗi lớp học phần.

### 3.4 Chuẩn hoá dữ liệu

- `id_student` là mã định danh nên chuyển sang chuỗi.
- `code_module`, `final_result` là biến phân loại nên chuyển sang `category`.
- Tách `code_presentation` thành `pres_year` và `pres_semester` (B = Feb, J = Oct).

Sau bước này xuất `data/cleaned/cleaned_dataset.csv`.

In [ ]:
df['id_student']    = df['id_student'].astype(str)
df['pres_year']     = df['code_presentation'].str[:4].astype(int)
df['pres_semester'] = df['code_presentation'].str[-1].map({'B': 'Feb', 'J': 'Oct'})
for c in ['code_module', 'pres_semester', 'final_result']:
    df[c] = df[c].astype('category')

df.to_csv('data/cleaned/cleaned_dataset.csv', index=False)
df.dtypes

# PART 4: EDA & FEATURE ENGINEERING

### 4.1 Phân tích đơn biến

Xem phân bố của target và 4 biến hành vi chính. **Skew** (độ lệch) ghi ở tiêu đề: dương = đuôi dài bên phải.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['at_risk'].value_counts().sort_index().plot.bar(ax=axes[0], color=['#4C9F70', '#D25C5C'], rot=0)
axes[0].set_title('at_risk (0 = an toàn, 1 = nguy cơ)')
df['final_result'].value_counts().plot.bar(ax=axes[1], color='#4C78A8', rot=0)
axes[1].set_title('final_result')
plt.tight_layout(); plt.show()

uni = ['total_clicks', 'active_days', 'clicks_before_start', 'avg_score']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, c in zip(axes, uni):
    df[c].plot.hist(bins=40, ax=ax, color='#4C78A8')
    ax.set_title(f'{c} (skew = {df[c].skew():.2f})')
plt.tight_layout(); plt.show()

**Nhận xét.**
- Target: **61% an toàn / 39% nguy cơ**, lệch nhẹ. Không cần sinh thêm mẫu (SMOTE), chỉ cần dùng `class_weight='balanced'` khi huấn luyện.
- `total_clicks` và `clicks_before_start` lệch phải mạnh: đa số sinh viên click ít, một số rất ít click rất nhiều. Hai biến này sẽ được biến đổi `log(1 + x)` ở phần 4.5.
- `avg_score` lệch trái, có một cụm tại 0 điểm, chính là các sinh viên chưa nộp bài.

### 4.2 Phân tích song biến (biến X và `at_risk`)

So sánh median của từng biến giữa nhóm an toàn và nhóm nguy cơ. Chênh lệch càng lớn thì biến càng phân biệt tốt.

In [ ]:
num = ['num_of_prev_attempts', 'studied_credits', 'date_registration',
       'total_clicks', 'active_days', 'n_resources', 'clicks_before_start',
       'n_assess_due', 'n_submitted', 'avg_score', 'n_late']
med = df.groupby('at_risk')[num].median().T
med.columns = ['median_an_toan', 'median_nguy_co']
med['chenh_lech'] = med['median_an_toan'] - med['median_nguy_co']
med.sort_values('chenh_lech', key=abs, ascending=False)

In [ ]:
rate = df.groupby('code_module', observed=True)['at_risk'].mean().sort_values()
ax = rate.plot.bar(figsize=(10, 4), color='#D25C5C', rot=0)
ax.axhline(df['at_risk'].mean(), color='black', ls='--', label='Trung bình')
ax.set_title('Tỷ lệ at_risk theo code_module'); ax.legend()
plt.tight_layout(); plt.show()

**Nhận xét.**
- Nhóm biến **hoạt động VLE** (`total_clicks`, `active_days`, `n_resources`, `clicks_before_start`) và **điểm** (`avg_score`) chênh lệch rõ nhất giữa hai nhóm: sinh viên nguy cơ click ít hơn khoảng 2,5 lần và điểm thấp hơn khoảng 17 điểm.
- `studied_credits` và `date_registration` có median gần như bằng nhau ở hai nhóm, tức là không phân biệt được nguy cơ. **Hai biến này bị loại khỏi mô hình** ở phần 5.
- Tỷ lệ nguy cơ khác nhau rõ giữa các môn, từ khoảng 22% (AAA) đến 45% (DDD), nên `code_module` được đưa vào mô hình làm biến điều khiển.

### 4.3 Phân tích đa biến (tương quan Spearman)

Dùng **Spearman** (tính trên thứ hạng) thay vì Pearson vì dữ liệu lệch mạnh và nhiều outlier. |ρ| ≥ 0.7 giữa hai biến X là dấu hiệu đa cộng tuyến.

In [ ]:
corr = df[num + ['dropout_no_activity', 'active_but_no_submission', 'at_risk']].corr(method='spearman')
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, annot_kws={'size': 8})
plt.title('Tương quan Spearman'); plt.tight_layout(); plt.show()

corr['at_risk'].drop('at_risk').sort_values()

**Nhận xét.**
- Tương quan âm mạnh nhất với `at_risk`: `active_days`, `total_clicks`, `avg_score`, `n_resources`. Hoạt động càng nhiều, điểm càng cao thì nguy cơ càng thấp.
- Tương quan dương (yếu hơn): `dropout_no_activity`, `num_of_prev_attempts`, `active_but_no_submission`.
- `studied_credits`, `date_registration` gần như không tương quan với `at_risk` (|ρ| < 0.06), khớp với nhận xét ở 4.2.
- Các cặp `total_clicks` – `active_days` – `n_resources` tương quan cao với nhau (ρ ≈ 0.76 – 0.89). Không bỏ biến nào vì chúng đo các khía cạnh khác nhau; thay vào đó tạo biến tỷ số `clicks_per_active_day` (4.5) và dùng ràng buộc L2 mặc định trong Logistic Regression.

### 4.4 Boxplot

Boxplot theo nhóm `at_risk` cho các biến hành vi. Điểm nằm ngoài râu là outlier theo quy tắc IQR: ngoài khoảng $[Q_1 - 1.5\,IQR,\ Q_3 + 1.5\,IQR]$.

In [ ]:
box = ['total_clicks', 'active_days', 'n_resources', 'clicks_before_start', 'avg_score', 'n_late']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, c in zip(axes.flat, box):
    sns.boxplot(data=df, x='at_risk', y=c, ax=ax, color='#6BAED6', fliersize=2)
    ax.set_title(c)
plt.tight_layout(); plt.show()

q1, q3 = df[box].quantile(.25), df[box].quantile(.75)
iqr = q3 - q1
((df[box] < q1 - 1.5 * iqr) | (df[box] > q3 + 1.5 * iqr)).sum().rename('n_outlier_IQR')

**Nhận xét.**
- Ở mọi biến hoạt động VLE, hộp của nhóm nguy cơ nằm thấp hơn hẳn nhóm an toàn.
- Outlier của các biến click đều nằm ở đuôi phải: đó là sinh viên rất chăm, không phải lỗi dữ liệu. Nếu xoá theo IQR sẽ mất hàng nghìn dòng mang tín hiệu tốt, nên sẽ xử lý bằng cách **chặn tại phân vị 99** (phần 4.6).
- `avg_score` có outlier ở phía dưới (điểm thấp) và đó chính là tín hiệu nguy cơ, nên giữ nguyên.

### 4.5 Tạo biến mới (Feature Engineering)

| Biến mới | Công thức | Ý nghĩa |
|---|---|---|
| `n_missed` | `n_assess_due − n_submitted` | Số bài bỏ lỡ, sinh viên không theo kịp lịch. |
| `submission_rate` | `n_submitted / n_assess_due` | Tỷ lệ nộp bài, so sánh được giữa môn ít bài và nhiều bài. |
| `late_rate` | `n_late / n_submitted` | Tỷ lệ nộp trễ. |
| `clicks_per_active_day` | `total_clicks / active_days` | Cường độ học mỗi ngày; tách "học đều" và "học dồn", giảm trùng lặp giữa `total_clicks` và `active_days`. |
| `is_repeater` | `num_of_prev_attempts > 0` | 87% sinh viên chưa học lại; điểm khác biệt lớn nhất là có/không học lại, không phải số lần. |
| `log_total_clicks` | `log(1 + total_clicks)` | Giảm độ lệch phải. Dùng `log(1+x)` vì nhiều sinh viên có 0 click. |
| `log_clicks_before_start` | `log(1 + clicks_before_start)` | Như trên. |

Phép chia cho 0 (chưa có bài đến hạn / chưa nộp / chưa hoạt động) được gán bằng 0.

In [ ]:
fe = df.copy()
fe['n_missed']                = fe['n_assess_due'] - fe['n_submitted']
fe['submission_rate']         = (fe['n_submitted'] / fe['n_assess_due']).fillna(0)
fe['late_rate']               = (fe['n_late'] / fe['n_submitted']).fillna(0)
fe['clicks_per_active_day']   = (fe['total_clicks'] / fe['active_days']).fillna(0)
fe['is_repeater']             = (fe['num_of_prev_attempts'] > 0).astype(int)
fe['log_total_clicks']        = np.log1p(fe['total_clicks'])
fe['log_clicks_before_start'] = np.log1p(fe['clicks_before_start'])

NEW = ['n_missed', 'submission_rate', 'late_rate', 'clicks_per_active_day',
       'is_repeater', 'log_total_clicks', 'log_clicks_before_start']
fe[NEW].describe().round(2).T

**Kiểm tra đa cộng tuyến bằng VIF.** VIF = 1 / (1 − R²), với R² là mức một biến được dự đoán bởi các biến còn lại. VIF < 5: an toàn; 5 – 10: cảnh báo; > 10: trùng lặp mạnh.

In [ ]:
from sklearn.linear_model import LinearRegression

FEATURES_NUM = ['log_total_clicks', 'active_days', 'n_resources', 'log_clicks_before_start',
                'n_assess_due', 'n_submitted', 'avg_score', 'n_late',
                'n_missed', 'submission_rate', 'late_rate', 'clicks_per_active_day',
                'is_repeater', 'dropout_no_activity', 'active_but_no_submission']

def vif(X):
    return pd.Series({c: 1 / (1 - LinearRegression().fit(X.drop(columns=c), X[c])
                                   .score(X.drop(columns=c), X[c]))
                      for c in X.columns}).sort_values(ascending=False).round(2)

vif(fe[FEATURES_NUM].drop(columns='n_assess_due'))

**Nhận xét.** `n_assess_due` không đưa vào VIF vì `n_missed = n_assess_due − n_submitted` là quan hệ tuyến tính chính xác (VIF vô hạn). Các biến VIF cao (`n_late`, `submission_rate`, `active_but_no_submission`) đều được tạo trực tiếp từ các biến khác nên trùng lặp là đương nhiên. Vẫn giữ lại vì:
- Random Forest không bị ảnh hưởng bởi đa cộng tuyến.
- Logistic Regression dùng ràng buộc L2 (mặc định của `sklearn`) giữ các hệ số nhỏ và ổn định.

### 4.6 Xử lý outlier (làm sạch lần 2)

Từ boxplot ở 4.4: outlier là sinh viên rất chăm, không phải lỗi, nên **giữ dòng và chặn giá trị tại phân vị 99 (P99)** thay vì xoá. Chỉ áp dụng cho các biến click còn ở thang gốc (`active_days`, `n_resources`, `clicks_per_active_day`); hai biến click tổng đã được nén đuôi bằng log ở 4.5. `avg_score` giữ nguyên vì đuôi thấp là tín hiệu nguy cơ.

Sau bước này xuất `data/eda/features.csv`.

In [ ]:
CAP = ['active_days', 'n_resources', 'clicks_per_active_day']
p99 = fe[CAP].quantile(.99)
print('Số dòng bị chặn:', (fe[CAP] > p99).sum().to_dict())
fe[CAP] = fe[CAP].clip(upper=p99, axis=1)

fe.to_csv('data/eda/features.csv', index=False)
fe[CAP].describe().round(1).T

# PART 5: MÔ HÌNH & ĐÁNH GIÁ

### 5.1 Chia dữ liệu theo thời gian

3 kỳ cũ (2013B, 2013J, 2014B) làm **tập huấn luyện**, kỳ mới nhất (2014J) làm **tập kiểm tra**. Mô hình học từ quá khứ và được kiểm tra trên tương lai, giống như khi trường dùng dữ liệu các kỳ trước để dự đoán kỳ sắp tới. Chia ngẫu nhiên sẽ để sinh viên cùng lớp rơi vào cả train lẫn test, cho kết quả lạc quan quá mức.

Biến đưa vào mô hình: 15 biến số (`FEATURES_NUM` ở 4.5) + 2 biến phân loại (`code_module`, `pres_semester`). Loại bỏ: `id_student`, `final_result` (nguồn target), `cutoff_day`, `code_presentation`, `num_of_prev_attempts` (đã thành `is_repeater`), `total_clicks` và `clicks_before_start` (đã thay bằng bản log), `studied_credits` và `date_registration` (không phân biệt được nguy cơ, xem 4.2).

In [ ]:
FEATURES_CAT = ['code_module', 'pres_semester']
FEATURES = FEATURES_NUM + FEATURES_CAT

test = fe['code_presentation'] == '2014J'
X_train, y_train = fe.loc[~test, FEATURES], fe.loc[~test, 'at_risk']
X_test,  y_test  = fe.loc[test,  FEATURES], fe.loc[test,  'at_risk']

print(f'Train: {len(X_train):,} dòng, at_risk = {y_train.mean():.3f}')
print(f'Test : {len(X_test):,} dòng, at_risk = {y_test.mean():.3f}')

### 5.2 Pipeline + 2 mô hình

`ColumnTransformer` chuẩn hoá biến số (`StandardScaler`) và one-hot biến phân loại; chỉ fit trên train nên không rò rỉ thông tin từ test. Hai mô hình:
- **Logistic Regression**: mô hình tuyến tính, có ràng buộc L2, dễ diễn giải qua hệ số.
- **Random Forest**: phi tuyến, bắt được tương tác giữa các biến.

Cả hai dùng `class_weight='balanced'` để phạt việc bỏ sót sinh viên nguy cơ nặng hơn.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

prep = ColumnTransformer([
    ('num', StandardScaler(), FEATURES_NUM),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), FEATURES_CAT),
])
models = {
    'Logistic Regression': make_pipeline(prep, LogisticRegression(
        class_weight='balanced', max_iter=2000, random_state=RNG)),
    'Random Forest': make_pipeline(prep, RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight='balanced',
        n_jobs=-1, random_state=RNG)),
}
for m in models.values():
    m.fit(X_train, y_train)

### 5.3 Đánh giá trên tập test (2014J, ngưỡng 0.5)

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)

rows = []
for name, m in models.items():
    yp, ypp = m.predict(X_test), m.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, yp).ravel()
    rows.append({'model': name,
                 'accuracy': accuracy_score(y_test, yp),
                 'precision': precision_score(y_test, yp),
                 'recall': recall_score(y_test, yp),
                 'f1': f1_score(y_test, yp),
                 'roc_auc': roc_auc_score(y_test, ypp),
                 'pr_auc': average_precision_score(y_test, ypp),
                 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp})
pd.DataFrame(rows).set_index('model').round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for name, m in models.items():
    RocCurveDisplay.from_estimator(m, X_test, y_test, name=name, ax=axes[1])
    PrecisionRecallDisplay.from_estimator(m, X_test, y_test, name=name, ax=axes[2])
ConfusionMatrixDisplay.from_estimator(models['Random Forest'], X_test, y_test,
                                      display_labels=['An toàn', 'Nguy cơ'], cmap='Blues',
                                      colorbar=False, ax=axes[0])
axes[0].set_title('Random Forest — Confusion Matrix')
axes[1].plot([0, 1], [0, 1], 'k--'); axes[1].set_title('ROC')
axes[2].axhline(y_test.mean(), ls='--', color='gray', label='Baseline')
axes[2].set_title('Precision-Recall'); axes[2].legend()
plt.tight_layout(); plt.show()

### 5.4 Chọn ngưỡng dự đoán

Ngưỡng 0.5 chưa chắc là tốt nhất. Hạ ngưỡng sẽ bắt được nhiều sinh viên nguy cơ hơn (Recall tăng) nhưng nhiều cảnh báo nhầm hơn (Precision giảm).

In [ ]:
proba = models['Random Forest'].predict_proba(X_test)[:, 1]
pd.DataFrame([{'threshold': t,
               'precision': precision_score(y_test, proba >= t),
               'recall': recall_score(y_test, proba >= t),
               'n_canh_bao': int((proba >= t).sum())}
              for t in [0.3, 0.4, 0.5, 0.6]]).round(3)

**Nhận xét.**
- Hai mô hình có ROC-AUC (≈ 0.84) và PR-AUC (≈ 0.79) gần như bằng nhau, đều cao hơn hẳn mức đoán ngẫu nhiên (PR-AUC baseline = tỷ lệ nguy cơ trên test = 0.343).
- Logistic Regression có Precision và Accuracy cao hơn; Random Forest có Recall cao hơn (bỏ sót 890 sinh viên nguy cơ so với 1.025 của Logistic Regression).
- Với Random Forest, hạ ngưỡng từ 0.5 xuống 0.4 tăng Recall từ 0.69 lên 0.79, đổi lại Precision giảm từ 0.66 xuống 0.59.

### 5.5 Kết luận

**Câu hỏi 1.** Mức độ tương tác trên VLE (`active_days`, `total_clicks`, `n_resources`, `clicks_before_start`) và điểm các bài đã làm (`avg_score`) là những yếu tố dự báo mạnh nhất. Sinh viên không nộp bài và không vào VLE trước mốc cảnh báo gần như chắc chắn trượt hoặc rút (99%). Số tín chỉ và ngày đăng ký gần như không có tác dụng dự báo.

**Câu hỏi 2.** Chọn **Random Forest** làm mô hình chính vì Recall cao hơn, phù hợp mục tiêu không bỏ sót sinh viên nguy cơ; dùng Logistic Regression để kiểm tra chéo và diễn giải qua hệ số. Nếu muốn bắt khoảng 80% sinh viên nguy cơ thì dùng ngưỡng **0.4** (Recall 0.79); nếu nguồn lực tư vấn ít thì giữ ngưỡng 0.5 (Recall 0.69, ít cảnh báo nhầm hơn).

**Hạn chế.**
1. Chỉ có 4 kỳ dữ liệu, chưa bắt được xu hướng dài hạn.
2. Xác suất `predict_proba` chưa được hiệu chuẩn (calibration) nên chưa dùng trực tiếp làm "xác suất rủi ro".
3. Chưa kiểm tra công bằng theo nhóm sinh viên (fairness audit).
4. Ngưỡng P99 khi chặn outlier được tính trên toàn bộ dữ liệu trước khi chia train/test.